# FINS5557 Week 5 — AI Tools in Credit Scoring
**UNSW Business School | Hric & Lin, 2026**

---

This notebook implements the Week 5 Case Study: AI Tools in Credit Scoring,
progressing from traditional logistic scorecards through gradient boosting with
alternative data to neural network architectures and explainability tools.

| § | Topic | Key output |
|---|---|---|
| 1 | Traditional Logistic Scorecard (WoE/IV) | Feature IV ranking, scorecard performance |
| 2 | XGBoost with Alternative Data | AUC uplift from open banking signals |
| 3 | MLP Neural Network | Non-linear credit model |
| 4 | Model Comparison | ROC curves, Gini, KS statistic |
| 5 | SHAP Explainability | SR 11-7 feature contributions + adverse action notice |
| 6 | PSI Drift Monitoring | Population stability, APRA CPS 220 retrain trigger |
| 7 | Gemini: Alternative Data Narrative | Structured extraction from applicant text |

**Dataset:** 10,000 synthetic credit applicants (8% default rate) with traditional
bureau features plus open banking / alternative data signals, reflecting a realistic
Australian ADI (Authorised Deposit-taking Institution) credit portfolio.

**API note (§7):** Uses Google Gemini 1.5 Flash. Add `GOOGLE_API_KEY` to Colab
Secrets (key icon in sidebar) for live analysis. Mock path runs automatically.


In [ ]:
# Setup -- install and import required packages
# !pip install shap --quiet  # uncomment if shap not installed

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings, json, sys
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve
from sklearn.preprocessing import StandardScaler
from scipy.stats import ks_2samp
warnings.filterwarnings('ignore')

try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print('shap not installed. Section 5 will use permutation importance instead.')

plt.rcParams.update({'figure.dpi': 120, 'font.family': 'DejaVu Sans',
                     'axes.spines.top': False, 'axes.spines.right': False})
NAVY   = '#003366'
RED    = '#C8102E'
GREEN  = '#1A7A4A'
ORANGE = '#D4820A'
PURPLE = '#6B21A8'

# ============================================================
# Synthetic Credit Dataset  -- 10,000 applicants, ~8% default
# Reflects a realistic Australian ADI credit portfolio with
# traditional bureau features + open banking alternative data.
# ============================================================
np.random.seed(2025)
N = 10_000

# Traditional bureau features
credit_score      = np.clip(np.random.normal(680, 80, N), 300, 850).astype(int)
income_k          = np.clip(np.random.lognormal(4.2, 0.5, N), 25, 500)
employment_years  = np.clip(np.random.exponential(4, N), 0, 40)
debt_to_income    = np.clip(np.random.beta(2, 5, N) * 0.8, 0, 0.8)
num_credit_lines  = np.clip(np.random.poisson(3, N), 0, 15).astype(int)
previous_defaults = (np.random.rand(N) < 0.08).astype(int)
loan_amount_k     = np.clip(np.random.lognormal(3.5, 0.8, N), 1, 200)

# Alternative data features (open banking / digital signals)
util_payment_score = np.clip(np.random.beta(7, 2, N), 0, 1)      # utility payment regularity
rental_months      = np.clip(np.random.exponential(18, N), 0, 72) # months of on-time rent
tx_velocity        = np.clip(np.random.poisson(45, N), 1, 200)    # transactions/month
balance_stability  = np.clip(np.random.beta(4, 3, N), 0, 1)       # bank balance stability
gig_income_pct     = np.clip(np.random.beta(1, 8, N), 0, 0.6)     # proportion of gig income
digital_engagement = np.clip(np.random.beta(5, 3, N), 0, 1)       # digital banking engagement

# Default label: correlated with features; ~8% base rate
logit = (
    -4.50  - 0.015  * (credit_score - 680)
    + 0.80 * previous_defaults
    + 2.50 * debt_to_income
    - 0.030 * income_k
    - 0.050 * employment_years
    - 0.80  * util_payment_score
    - 0.60  * balance_stability
    + 0.40  * gig_income_pct
    + np.random.normal(0, 0.3, N)
)
default_prob = 1 / (1 + np.exp(-logit))
default      = (np.random.rand(N) < default_prob).astype(int)

TRADITIONAL = ['credit_score', 'income_k', 'employment_years', 'debt_to_income',
               'num_credit_lines', 'previous_defaults', 'loan_amount_k']
ALT_DATA    = ['util_payment_score', 'rental_months', 'tx_velocity',
               'balance_stability', 'gig_income_pct', 'digital_engagement']
ALL_FEAT    = TRADITIONAL + ALT_DATA

df = pd.DataFrame({
    'credit_score': credit_score,  'income_k': income_k,
    'employment_years': employment_years, 'debt_to_income': debt_to_income,
    'num_credit_lines': num_credit_lines, 'previous_defaults': previous_defaults,
    'loan_amount_k': loan_amount_k, 'util_payment_score': util_payment_score,
    'rental_months': rental_months, 'tx_velocity': tx_velocity,
    'balance_stability': balance_stability, 'gig_income_pct': gig_income_pct,
    'digital_engagement': digital_engagement, 'default': default,
})

# Stratified 80/20 split
X_trad = df[TRADITIONAL].values
X_all  = df[ALL_FEAT].values
y      = df['default'].values
X_tr_t, X_te_t, y_train, y_test = train_test_split(
    X_trad, y, test_size=0.2, random_state=42, stratify=y)
X_tr_a, X_te_a, _, _ = train_test_split(
    X_all,  y, test_size=0.2, random_state=42, stratify=y)

print(f'Dataset: {N:,} applicants | Default rate: {y.mean():.1%}')
print(f'Train: {len(y_train):,} | Test: {len(y_test):,}')
print(f'Traditional features: {TRADITIONAL}')
print(f'Alternative data features: {ALT_DATA}')


---
## §1 — Traditional Logistic Regression Scorecard (WoE/IV)

**Seminar reference:** Week 5, §4.1–4.2 — Alternative Data and Model Evaluation

The logistic regression **scorecard** is the regulatory gold standard for retail credit
decisioning. Its transparency makes it auditable under SR 11-7 and explainable under the
NCA s124 adverse action notice requirement.

**Weight of Evidence (WoE)** transforms raw feature values into a log-odds scale:

$$\text{WoE}_i = \ln\!\left(\frac{\text{Distribution of Goods}_i}{\text{Distribution of Bads}_i}\right) \tag{5.4}$$

**Information Value (IV)** aggregates predictive power across bins:

$$\text{IV} = \sum_{i=1}^{K} (\text{Dist. Goods}_i - \text{Dist. Bads}_i) \cdot \text{WoE}_i \tag{5.5}$$

| IV Range | Interpretation |
|---|---|
| < 0.02 | Negligible predictive power |
| 0.02 – 0.10 | Weak |
| 0.10 – 0.30 | Medium |
| > 0.30 | Strong |

*[Siddiqi, 2006]*


In [ ]:
# Section 1: WoE/IV feature screening + logistic regression scorecard
# Implements the standard credit scorecard workflow from the seminar.

def compute_iv(feature_vals, target, n_bins=5):
    # Bin the feature into n equal-frequency bins, compute WoE and IV per bin.
    # Returns: iv (float), woe_table (DataFrame)
    df_tmp = pd.DataFrame({'x': feature_vals, 'y': target})
    df_tmp['bin'] = pd.qcut(df_tmp['x'], q=n_bins, duplicates='drop')
    total_good = (df_tmp['y'] == 0).sum()
    total_bad  = (df_tmp['y'] == 1).sum()
    rows = []
    for bin_label, group in df_tmp.groupby('bin', observed=True):
        n_good = (group['y'] == 0).sum()
        n_bad  = (group['y'] == 1).sum()
        dist_g = max(n_good / total_good, 1e-6)
        dist_b = max(n_bad  / total_bad,  1e-6)
        woe    = np.log(dist_g / dist_b)
        iv_i   = (dist_g - dist_b) * woe
        rows.append({'bin': str(bin_label), 'n_good': n_good, 'n_bad': n_bad,
                     'dist_good': dist_g, 'dist_bad': dist_b, 'woe': woe, 'iv': iv_i})
    woe_table = pd.DataFrame(rows)
    return woe_table['iv'].sum(), woe_table

# Compute IV for every feature
print("Information Value (IV) -- Feature Ranking")
print("=" * 55)
iv_results = {}
for feat in ALL_FEAT:
    vals = df[feat].values
    iv_val, _ = compute_iv(vals, df['default'].values, n_bins=5)
    iv_results[feat] = iv_val
    flag = (
        'STRONG' if iv_val > 0.30 else
        'MEDIUM' if iv_val > 0.10 else
        'WEAK'   if iv_val > 0.02 else
        'NEGLIGIBLE'
    )
    data_type = 'Traditional' if feat in TRADITIONAL else 'Alt Data  '
    print(f"  {feat:25s} [{data_type}] IV = {iv_val:.3f}  ({flag})")

# Train logistic regression on WoE-binned traditional features
# (We use StandardScaler as a simple proxy for WoE normalisation)
scaler_trad = StandardScaler()
X_tr_sc = scaler_trad.fit_transform(X_tr_t)
X_te_sc = scaler_trad.transform(X_te_t)

lr_model = LogisticRegression(max_iter=500, C=0.5, random_state=42)
lr_model.fit(X_tr_sc, y_train)
y_pred_lr = lr_model.predict_proba(X_te_sc)[:, 1]
auc_lr    = roc_auc_score(y_test, y_pred_lr)
gini_lr   = 2 * auc_lr - 1

print(f"\nLogistic Regression Scorecard (traditional features only):")
print(f"  AUC = {auc_lr:.4f} | Gini = {gini_lr:.4f}")

# Visualise: IV bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

feat_names = list(iv_results.keys())
iv_vals    = [iv_results[f] for f in feat_names]
bar_cols   = [NAVY if f in TRADITIONAL else RED for f in feat_names]
bars = axes[0].barh(feat_names, iv_vals, color=bar_cols, alpha=0.85)
axes[0].axvline(0.10, color='orange', linestyle='--', lw=1.5, label='Weak/Medium (0.10)')
axes[0].axvline(0.30, color='green',  linestyle='--', lw=1.5, label='Medium/Strong (0.30)')
axes[0].set_xlabel('Information Value (IV)')
axes[0].set_title('Feature IV Ranking\n(Traditional vs. Alternative Data)', color=NAVY)
axes[0].legend(fontsize=8)
blue_patch = mpatches.Patch(color=NAVY, label='Traditional')
red_patch  = mpatches.Patch(color=RED,  label='Alternative data')
axes[0].legend(handles=[blue_patch, red_patch] +
               [plt.Line2D([0],[0], color='orange', linestyle='--', label='IV=0.10'),
                plt.Line2D([0],[0], color='green',  linestyle='--', label='IV=0.30')],
               fontsize=7)

# WoE table for top feature (credit_score)
top_feat = max(iv_results, key=iv_results.get)
_, woe_tbl = compute_iv(df[top_feat].values, df['default'].values, n_bins=5)
axes[1].bar(range(len(woe_tbl)), woe_tbl['woe'],
            color=[RED if w < 0 else NAVY for w in woe_tbl['woe']], alpha=0.85)
axes[1].axhline(0, color='black', lw=0.8)
axes[1].set_xticks(range(len(woe_tbl)))
axes[1].set_xticklabels([f'Bin {i+1}' for i in range(len(woe_tbl))], fontsize=9)
axes[1].set_ylabel('WoE')
axes[1].set_title(f'WoE per Bin: {top_feat}\n(IV = {iv_results[top_feat]:.3f})', color=NAVY)
axes[1].set_xlabel('Score Bin (low to high)')

plt.tight_layout()
plt.show()
print("\nInterpretation: Negative WoE = higher-risk bin; Positive WoE = lower-risk bin.")


---
## §2 — XGBoost with Alternative Data

**Seminar reference:** Week 5, §4.2–4.4

Gradient boosting (XGBoost) captures non-linear interactions between features without
manual WoE binning. Its tree structure naturally handles mixed continuous/categorical
inputs. The key question is: how much predictive lift does **alternative data** add?

**DDF Pipeline Mapping:**

| DDF Station | Credit Scoring Role |
|---|---|
| Station 1: ETL | Ingest bureau data + open banking CDR feeds |
| Station 2: Feature Engineering | WoE binning (traditional) + normalisation (alt data) |
| Station 3: Model Design | XGBoost on full feature set |
| Station 4: Implementation | Score production + monitoring dashboard |

**Alternative data features used:**
- `util_payment_score` — regularity of utility payments (open banking-derived)
- `rental_months` — months of on-time rental payments (property data feed)
- `tx_velocity` — transactions/month (open banking CDR)
- `balance_stability` — bank balance stability index (open banking CDR)
- `gig_income_pct` — proportion of income from gig economy (CDR income classification)
- `digital_engagement` — digital banking engagement score (internal bank signal)

**CDR note:** In Australia, all six alternative features above can be legally obtained
with consumer consent under the Consumer Data Right (CDR, 2020).


In [ ]:
# Section 2: XGBoost credit scoring -- traditional only vs. traditional + alt data
# Demonstrates the predictive lift from open banking alternative data.

scaler_all = StandardScaler()
X_tr_all_sc = scaler_all.fit_transform(X_tr_a)
X_te_all_sc = scaler_all.transform(X_te_a)

# Model 1: XGBoost, traditional features only
xgb_trad = GradientBoostingClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.08, min_samples_leaf=20,
    subsample=0.8, random_state=42)
xgb_trad.fit(X_tr_sc, y_train)
y_pred_xgb_t = xgb_trad.predict_proba(X_te_sc)[:, 1]
auc_xgb_t    = roc_auc_score(y_test, y_pred_xgb_t)
gini_xgb_t   = 2 * auc_xgb_t - 1

# Model 2: XGBoost, traditional + alternative data
xgb_all = GradientBoostingClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.08, min_samples_leaf=20,
    subsample=0.8, random_state=42)
xgb_all.fit(X_tr_all_sc, y_train)
y_pred_xgb_a = xgb_all.predict_proba(X_te_all_sc)[:, 1]
auc_xgb_a    = roc_auc_score(y_test, y_pred_xgb_a)
gini_xgb_a   = 2 * auc_xgb_a - 1

print("XGBoost Performance Comparison")
print("=" * 45)
print(f"  Traditional features only:   AUC = {auc_xgb_t:.4f}  Gini = {gini_xgb_t:.4f}")
print(f"  + Alternative data:          AUC = {auc_xgb_a:.4f}  Gini = {gini_xgb_a:.4f}")
print(f"  Gini uplift from alt data:   +{gini_xgb_a - gini_xgb_t:.4f}")

# Feature importance for the alt-data model
feat_importance = pd.Series(
    xgb_all.feature_importances_, index=ALL_FEAT
).sort_values(ascending=False)

print("\nTop 10 Feature Importances (XGBoost + alt data):")
for feat, imp in feat_importance.head(10).items():
    data_type = '[Traditional]' if feat in TRADITIONAL else '[Alt Data  ]'
    bar       = '#' * int(imp * 200)
    print(f"  {feat:25s} {data_type} {imp:.4f}  {bar}")

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Feature importance
feat_cols = [NAVY if f in TRADITIONAL else RED for f in feat_importance.index]
axes[0].barh(feat_importance.index, feat_importance.values, color=feat_cols, alpha=0.85)
axes[0].set_xlabel('Feature Importance (XGBoost)')
axes[0].set_title('Feature Importance: XGBoost + Alt Data\n(red = alternative data features)', color=NAVY)
blue_p = mpatches.Patch(color=NAVY, label='Traditional')
red_p  = mpatches.Patch(color=RED,  label='Alt Data')
axes[0].legend(handles=[blue_p, red_p], fontsize=8)

# Panel 2: Score distribution by default status
score_good = y_pred_xgb_a[y_test == 0]
score_bad  = y_pred_xgb_a[y_test == 1]
bins = np.linspace(0, 1, 41)
axes[1].hist(score_good, bins=bins, color=NAVY, alpha=0.6, density=True, label='Non-default (good)')
axes[1].hist(score_bad,  bins=bins, color=RED,  alpha=0.7, density=True, label='Default (bad)')
axes[1].set_xlabel('Predicted Default Probability')
axes[1].set_ylabel('Density')
axes[1].set_title(f'Score Separation by Default Status\nGini = {gini_xgb_a:.3f}', color=NAVY)
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()


---
## §3 — MLP Neural Network Credit Scoring

**Seminar reference:** Week 5, §4.2 — Neural Networks for Credit Scoring

A **Multi-Layer Perceptron (MLP)** captures higher-order non-linear interactions
that XGBoost cannot model through its tree structure. The trade-off is reduced
interpretability — the "black box" problem that makes neural networks challenging
under SR 11-7 without a post-hoc explainability layer (Section 5).

**Wide & Deep architecture** (Cheng et al., 2016) addresses this by combining:
- **Wide component** (linear): interpretable, auditable by regulators
- **Deep component** (DNN): captures non-linear feature interactions

The `sklearn.MLPClassifier` below implements the deep component; in production,
the wide component's weights are reported directly to SR 11-7 model validators.


In [ ]:
# Section 3: MLP Neural Network credit scoring
# Implements a feedforward neural network using sklearn.
# Traditional features only (baseline) and full feature set (with alt data).

# Model 3: MLP, traditional features
mlp_trad = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32),
    activation='relu', solver='adam',
    alpha=1e-3, batch_size=256,
    learning_rate='adaptive', max_iter=300,
    early_stopping=True, validation_fraction=0.1,
    random_state=42)
mlp_trad.fit(X_tr_sc, y_train)
y_pred_mlp_t = mlp_trad.predict_proba(X_te_sc)[:, 1]
auc_mlp_t    = roc_auc_score(y_test, y_pred_mlp_t)
gini_mlp_t   = 2 * auc_mlp_t - 1

# Model 4: MLP, traditional + alternative data
mlp_all = MLPClassifier(
    hidden_layer_sizes=(256, 128, 64, 32),
    activation='relu', solver='adam',
    alpha=1e-3, batch_size=256,
    learning_rate='adaptive', max_iter=300,
    early_stopping=True, validation_fraction=0.1,
    random_state=42)
mlp_all.fit(X_tr_all_sc, y_train)
y_pred_mlp_a = mlp_all.predict_proba(X_te_all_sc)[:, 1]
auc_mlp_a    = roc_auc_score(y_test, y_pred_mlp_a)
gini_mlp_a   = 2 * auc_mlp_a - 1

print("MLP Neural Network Performance")
print("=" * 45)
print(f"  Traditional features only:   AUC = {auc_mlp_t:.4f}  Gini = {gini_mlp_t:.4f}")
print(f"  + Alternative data:          AUC = {auc_mlp_a:.4f}  Gini = {gini_mlp_a:.4f}")
print(f"  Iterations (convergence):    {mlp_all.n_iter_}")

# Architecture summary
print("\nMLP Architecture (full feature set):")
print(f"  Input layer:  {len(ALL_FEAT)} features")
for i, size in enumerate([256, 128, 64, 32]):
    print(f"  Hidden {i+1}:     {size} neurons (ReLU)")
print("  Output:        1 neuron (sigmoid) -> P(default)")
print("\nRegularisation: alpha=1e-3 (L2), early stopping, validation_fraction=0.10")
print("SR 11-7 note: MLP requires SHAP post-hoc explanations for conceptual soundness.")


---
## §4 — Model Comparison: ROC Curves, Gini, KS Statistic

**Seminar reference:** Week 5, §4.3 — Model Evaluation Framework

**Gini Coefficient (Eq 5.1):** rank-ordering ability of the model

$$\text{Gini} = 2 \cdot \text{AUC} - 1 \tag{5.1}$$

**Kolmogorov-Smirnov Statistic (Eq 5.2):** maximum separation between cumulative
good and bad score distributions

$$KS = \max_{t \in [0,1]} \left|F_G(t) - F_B(t)\right| \tag{5.2}$$

| Metric | Logistic (trad) | XGBoost (trad) | XGBoost (+alt) | MLP (trad) | MLP (+alt) |
|---|---|---|---|---|---|
| **AUC** | ~0.72 | ~0.78 | ~0.82 | ~0.79 | ~0.84 |
| **Gini** | ~0.44 | ~0.56 | ~0.64 | ~0.58 | ~0.68 |
| **SR 11-7 Load** | Low | Medium | Medium | High | High |

*Actual values computed in this section from the synthetic dataset.*

**Regulatory-optimal choice for an Australian ADI:** XGBoost + alternative data
(Gini ~0.64, medium validation overhead). The MLP achieves +4 Gini points at the
cost of significantly higher SR 11-7 independent model validation effort.


In [ ]:
# Section 4: Model comparison -- ROC curves, Gini, KS statistic
# Compares all five model configurations on the held-out test set.

def ks_statistic(y_true, y_score):
    # Compute KS statistic: max separation between cumulative good/bad distributions.
    scores_good = np.sort(y_score[y_true == 0])
    scores_bad  = np.sort(y_score[y_true == 1])
    n_good = len(scores_good)
    n_bad  = len(scores_bad)
    all_scores = np.sort(np.unique(np.concatenate([scores_good, scores_bad])))
    ks = 0.0
    for t in all_scores:
        cdf_g = (scores_good <= t).sum() / n_good
        cdf_b = (scores_bad  <= t).sum() / n_bad
        ks = max(ks, abs(cdf_g - cdf_b))
    return ks

models = {
    'Logistic (trad)':     (y_pred_lr,      PURPLE),
    'XGBoost (trad)':      (y_pred_xgb_t,   NAVY),
    'XGBoost (+alt)':      (y_pred_xgb_a,   GREEN),
    'MLP (trad)':          (y_pred_mlp_t,   ORANGE),
    'MLP (+alt)':          (y_pred_mlp_a,   RED),
}

print("Model Comparison -- Test Set Performance")
print("=" * 65)
print(f"{'Model':22s}  {'AUC':>7s}  {'Gini':>7s}  {'KS (%)':>8s}  SR 11-7")
print("-" * 65)
sr117 = {'Logistic (trad)':'Low     ', 'XGBoost (trad)':'Medium  ',
         'XGBoost (+alt)': 'Medium  ', 'MLP (trad)':    'High    ',
         'MLP (+alt)':     'High    '}
results = {}
for name, (y_pred, col) in models.items():
    auc  = roc_auc_score(y_test, y_pred)
    gini = 2 * auc - 1
    ks   = ks_statistic(y_test, y_pred)
    results[name] = {'auc': auc, 'gini': gini, 'ks': ks, 'color': col, 'pred': y_pred}
    print(f"{name:22s}  {auc:>7.4f}  {gini:>7.4f}  {ks*100:>7.1f}%  {sr117[name]}")

print("=" * 65)

# Visualisation: 3 panels
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Panel 1: ROC curves
for name, r in results.items():
    fpr, tpr, _ = roc_curve(y_test, r['pred'])
    lw = 2.5 if 'alt' in name else 1.5
    ls = '-'  if 'alt' in name else '--'
    axes[0].plot(fpr, tpr, color=r['color'], lw=lw, linestyle=ls,
                 label=f"{name} (AUC={r['auc']:.3f})")
axes[0].plot([0,1],[0,1],'k:',lw=0.8, label='Random')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves -- All Models', color=NAVY)
axes[0].legend(fontsize=7)

# Panel 2: Gini comparison
names  = list(results.keys())
ginis  = [results[n]['gini'] for n in names]
cols   = [results[n]['color'] for n in names]
bars   = axes[1].bar(names, ginis, color=cols, alpha=0.85)
for b, g in zip(bars, ginis):
    axes[1].text(b.get_x() + b.get_width()/2, g + 0.005, f'{g:.3f}',
                 ha='center', fontsize=9)
axes[1].set_title('Gini Coefficient by Model\n(Eq 5.1: Gini = 2*AUC - 1)', color=NAVY)
axes[1].set_ylabel('Gini')
axes[1].set_ylim(0, 0.85)
axes[1].tick_params(axis='x', rotation=30)
axes[1].axhline(0.50, color='orange', linestyle=':', lw=1.2, label='Gini=0.50 (acceptable)')
axes[1].legend(fontsize=7)

# Panel 3: KS statistic comparison
ks_vals = [results[n]['ks'] * 100 for n in names]
bars    = axes[2].bar(names, ks_vals, color=cols, alpha=0.85)
for b, k in zip(bars, ks_vals):
    axes[2].text(b.get_x() + b.get_width()/2, k + 0.3, f'{k:.1f}%',
                 ha='center', fontsize=9)
axes[2].set_title('KS Statistic by Model\n(Eq 5.2: max|F_G - F_B|)', color=NAVY)
axes[2].set_ylabel('KS (%)')
axes[2].tick_params(axis='x', rotation=30)
axes[2].axhline(40, color='green', linestyle=':', lw=1.2, label='KS=40% (strong)')
axes[2].legend(fontsize=7)

plt.tight_layout()
plt.savefig('credit_model_comparison.png', bbox_inches='tight')
plt.show()

best = max(results, key=lambda n: results[n]['auc'])
print(f"\nBest model: {best} | AUC={results[best]['auc']:.4f} | Gini={results[best]['gini']:.4f} | KS={results[best]['ks']*100:.1f}%")


---
## §5 — SHAP Explainability (SR 11-7 Compliance)

**Seminar reference:** Week 5, §4.3 — SHAP for Model Documentation

**SHAP** (SHapley Additive exPlanations) decomposes each prediction into additive
feature contributions:

$$f(x) = \phi_0 + \sum_{i=1}^{M} \phi_i \cdot x_i$$

where $\phi_0$ is the base rate (mean prediction), and each $\phi_i$ is the
SHAP value for feature $i$ — the average marginal contribution of that feature
across all possible feature coalitions [Lundberg & Lee, 2017].

**SR 11-7 compliance:** The Federal Reserve's model risk guidance requires institutions
to demonstrate **how** a model produces its results (conceptual soundness). SHAP
provides a mathematically rigorous, post-hoc explanation that satisfies this requirement.

**Adverse action notice (NCA s124):** When an Australian consumer credit application
is declined, the lender must provide a reason. SHAP's top-N feature contributions
map directly to a legally compliant adverse action letter.


In [ ]:
# Section 5: SHAP explainability using the XGBoost + alt data model
# Generates global feature importance (summary plot) and individual prediction
# explanation (adverse action notice).

if SHAP_AVAILABLE:
    # TreeExplainer is fast for GradientBoostingClassifier
    explainer = shap.TreeExplainer(xgb_all)
    # Use a subset of 500 test samples for speed
    subset_idx = np.random.choice(len(X_te_a), size=min(500, len(X_te_a)), replace=False)
    X_te_subset = scaler_all.transform(X_te_a[subset_idx])
    shap_vals   = explainer.shap_values(X_te_subset)

    # Global summary plot
    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_vals, X_te_subset, feature_names=ALL_FEAT,
                      show=False, plot_size=(10, 6))
    plt.title('SHAP Summary Plot: XGBoost + Alternative Data\n'
              '(Mean absolute SHAP value = average impact on predicted default probability)',
              color=NAVY)
    plt.tight_layout()
    plt.show()

    # Individual adverse action notice for a high-risk applicant
    high_risk_idx = np.argmax(xgb_all.predict_proba(X_te_all_sc)[:, 1])
    x_applicant   = scaler_all.transform(X_te_a[high_risk_idx:high_risk_idx+1])
    shap_indiv    = explainer.shap_values(x_applicant)[0]
    shap_df = pd.DataFrame({'feature': ALL_FEAT, 'shap_value': shap_indiv,
                            'feature_value': X_te_a[high_risk_idx]})
    shap_df = shap_df.reindex(shap_df['shap_value'].abs().sort_values(ascending=False).index)

    default_prob_val = xgb_all.predict_proba(x_applicant)[0][1]
    decision = 'DECLINED' if default_prob_val > 0.35 else 'APPROVED'

    sep = '-' * 62
    print(sep)
    print('  ADVERSE ACTION NOTICE (NCA s124 compliant)')
    print(sep)
    print(f'  Decision: {decision} | P(default) = {default_prob_val:.3f}')
    print(f'  Base rate: {xgb_all.predict_proba(X_te_all_sc)[:, 1].mean():.3f}')
    print(sep)
    print('  Primary contributing factors:')
    for i, (_, row) in enumerate(shap_df.head(5).iterrows(), 1):
        direction = 'INCREASES default risk' if row['shap_value'] > 0 else 'REDUCES default risk'
        print(f"  {i}. {row['feature']:25s}: {direction} (SHAP={row['shap_value']:+.3f})")
    print(sep)
    print('  SR 11-7 note: SHAP decomposition satisfies conceptual soundness')
    print('  documentation requirement for individual prediction auditing.')
    print(sep)

else:
    # Fallback: use built-in feature importance
    print("SHAP not available -- using XGBoost built-in feature importance as proxy.")
    feat_imp = pd.Series(xgb_all.feature_importances_, index=ALL_FEAT).sort_values(ascending=False)
    print("\nFeature Importance (XGBoost, MDI):")
    for feat, imp in feat_imp.items():
        flag = '[Alt Data]' if feat in ALT_DATA else ''
        print(f"  {feat:25s}: {imp:.4f}  {flag}")
    print("\nInstall shap for full SHAP analysis: pip install shap")


---
## §6 — Population Stability Index (PSI): Drift Monitoring

**Seminar reference:** Week 5, §4.3 — PSI for Production Monitoring

The **Population Stability Index** detects distributional shift between the model
development population and the current monitoring population:

$$\text{PSI} = \sum_{i=1}^{N} (A_i - E_i) \cdot \ln\!\left(\frac{A_i}{E_i}\right) \tag{5.3}$$

where $A_i$ = actual proportion of applicants in bin $i$ (monitoring period),
$E_i$ = expected proportion (development period).

| PSI | Decision |
|---|---|
| < 0.10 | No significant change — monitor normally |
| 0.10 – 0.20 | Minor shift — investigate; increase monitoring frequency |
| > 0.20 | **Major shift — retrain model; notify APRA CPS 220** |

**APRA CPS 220 obligation:** Documented model monitoring plans must specify
the PSI trigger threshold for model retraining and the escalation procedure.

**Applied scenario:** RBA cash rate rising from 0.10% (April 2022) to 4.35%
(November 2023) shifted the income and debt-to-income distributions of applicants
materially, triggering PSI > 0.20 for several key features.


In [ ]:
# Section 6: Population Stability Index (PSI) drift detection
# Simulates a post-rate-rise population shift and computes PSI per feature.

def compute_psi(expected, actual, n_bins=10):
    # Compute PSI between expected (dev) and actual (monitoring) distributions.
    # Returns: psi (float), bins detail (list)
    all_vals = np.concatenate([expected, actual])
    bin_edges = np.percentile(expected, np.linspace(0, 100, n_bins + 1))
    bin_edges[0]  -= 1e-6
    bin_edges[-1] += 1e-6
    bins_e = []
    bins_a = []
    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
        pct_e = max(((expected >= lo) & (expected < hi)).mean(), 1e-6)
        pct_a = max(((actual   >= lo) & (actual   < hi)).mean(), 1e-6)
        bins_e.append(pct_e)
        bins_a.append(pct_a)
    psi = sum((a - e) * np.log(a / e) for a, e in zip(bins_a, bins_e))
    return psi, list(zip(bins_e, bins_a))

# Simulate post-rate-rise population (interest rate shock: +4.25% over 18 months)
np.random.seed(2026)
N_monitor = 3_000

# Income: compressed (fewer high earners applying after rate rise)
income_monitor = np.clip(np.random.lognormal(4.0, 0.45, N_monitor), 25, 500)
# DTI: increased (mortgage repayments higher)
dti_monitor    = np.clip(np.random.beta(3, 4, N_monitor) * 0.85, 0, 0.85)
# Credit score: slightly lower (more stress)
cs_monitor     = np.clip(np.random.normal(655, 85, N_monitor), 300, 850)
# Alternative data signals: more stable post-RBA shock (survivors effect)
util_monitor   = np.clip(np.random.beta(6, 2, N_monitor), 0, 1)
bal_monitor    = np.clip(np.random.beta(3, 3, N_monitor), 0, 1)

monitor_data = {
    'income_k':          (df['income_k'].values,         income_monitor,    RED    if True else NAVY),
    'debt_to_income':    (df['debt_to_income'].values,   dti_monitor,       RED    if True else NAVY),
    'credit_score':      (df['credit_score'].values,     cs_monitor,        ORANGE if True else NAVY),
    'util_payment_score':(df['util_payment_score'].values, util_monitor,    NAVY),
    'balance_stability': (df['balance_stability'].values, bal_monitor,      NAVY),
}

print("PSI Monitoring Report -- Post-Rate-Rise Population Shift")
print("=" * 60)
print(f"{'Feature':25s}  {'PSI':>7s}  Status")
print("-" * 60)
psi_results = {}
for feat, (dev, mon, col) in monitor_data.items():
    psi, _ = compute_psi(dev, mon, n_bins=10)
    psi_results[feat] = {'psi': psi, 'dev': dev, 'mon': mon, 'color': col}
    status = (
        'RETRAIN REQUIRED (APRA CPS 220 escalation)' if psi > 0.20 else
        'INVESTIGATE'                                  if psi > 0.10 else
        'STABLE'
    )
    print(f"  {feat:25s}  {psi:>7.4f}  {status}")
print("=" * 60)

# Visualisation: PSI bar chart + score distribution shift
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

feat_names_psi = list(psi_results.keys())
psi_vals = [psi_results[f]['psi'] for f in feat_names_psi]
bar_cols = [RED if p > 0.20 else (ORANGE if p > 0.10 else NAVY) for p in psi_vals]
bars = axes[0].bar(feat_names_psi, psi_vals, color=bar_cols, alpha=0.85)
axes[0].axhline(0.10, color=ORANGE, linestyle='--', lw=1.5, label='PSI=0.10 (investigate)')
axes[0].axhline(0.20, color=RED,    linestyle='--', lw=1.5, label='PSI=0.20 (retrain)')
axes[0].tick_params(axis='x', rotation=35)
axes[0].set_ylabel('PSI')
axes[0].set_title('Population Stability Index\n(Post-Rate-Rise vs Development Population)', color=NAVY)
axes[0].legend(fontsize=8)
for bar, val in zip(bars, psi_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 0.002,
                 f'{val:.3f}', ha='center', fontsize=8)

# Distribution comparison for highest-PSI feature
worst_feat = max(psi_results, key=lambda f: psi_results[f]['psi'])
dev_vals = psi_results[worst_feat]['dev']
mon_vals = psi_results[worst_feat]['mon']
x_range = np.linspace(min(dev_vals.min(), mon_vals.min()),
                      max(dev_vals.max(), mon_vals.max()), 50)
axes[1].hist(dev_vals, bins=40, density=True, color=NAVY, alpha=0.6, label='Development')
axes[1].hist(mon_vals, bins=40, density=True, color=RED,  alpha=0.6, label='Monitor (post-rate-rise)')
axes[1].set_xlabel(worst_feat)
axes[1].set_ylabel('Density')
axes[1].set_title(f'Distribution Shift: {worst_feat}\n'
                  f'PSI = {psi_results[worst_feat]["psi"]:.3f} -- '
                  f'{"RETRAIN" if psi_results[worst_feat]["psi"] > 0.20 else "INVESTIGATE"}',
                  color=NAVY)
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()


---
## §7 — Gemini API: Structured Alternative Data Extraction from Narrative

**Seminar reference:** Week 5, §4.1 — Alternative Data Sources; DDF Station 1: ETL

Many alternative data signals are embedded in unstructured text: bank statement
narratives, customer service transcripts, referral letters, and self-reported
income explanations. A language model can extract structured creditworthiness
signals from these narratives — a real-world example of **DDF Station 1 (ETL)**
applied to unstructured financial data.

**Case study:** An applicant submits a short written statement about their financial
situation. The four-stage Gemini pipeline extracts:
1. Income stability signals
2. Spending behaviour signals
3. Financial stress indicators
4. Recommended alternative data features with scores (0–1)

This output augments the traditional bureau score with qualitative signals that
open banking data alone may not capture (e.g., recent job change, upcoming major expense).

### Free Setup — Google Gemini API (2 minutes)

> **Step 1** — Get a free key at `https://aistudio.google.com/apikey`
> (no credit card; free tier: 15 RPM / 1M tokens/day)

> **Step 2** — In Colab: key icon (Secrets) → **+ Add new secret** →
> Name: `GOOGLE_API_KEY`, Value: paste key, toggle **Notebook access** ON

> **Step 3** — Run the cell — mock simulation runs by default; live Gemini activates automatically

**Model:** `gemini-1.5-flash` (free tier, sufficient for all course exercises)


In [ ]:
# Section 7: Multi-stage Gemini pipeline for alternative data extraction
# Free Colab Secrets setup -- mock simulation runs without any API key.

import sys, json

# Install google-generativeai if absent
try:
    import google.generativeai as genai
except ImportError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "google-generativeai", "-q"], check=True)
    import google.generativeai as genai

# Load API key: Colab Secrets -> env var -> mock fallback
def _load_api_key():
    """Return API key from Colab Secrets or OS environment, else None."""
    try:
        from google.colab import userdata
        key = userdata.get("GOOGLE_API_KEY")
        if key:
            print("API key loaded from Colab Secrets -- live Gemini path enabled.")
            return key
    except Exception:
        pass
    import os
    key = os.getenv("GOOGLE_API_KEY")
    if key:
        print("API key loaded from environment variable.")
        return key
    return None

GEMINI_API_KEY = _load_api_key()
USE_LIVE_API   = GEMINI_API_KEY is not None

if not USE_LIVE_API:
    print("No API key detected -- running mock simulation.")
    print("Follow the setup instructions above to enable live Gemini analysis.")

# Applicant narrative (simulated self-report submitted with credit application)
APPLICANT_NARRATIVE = """
My name is Sarah Chen and I'm applying for a $45,000 personal loan to consolidate debt
and cover costs of a home office renovation for my remote work setup.

I've been working as a freelance data analyst for the past three years, with my main
clients being two tech companies on 6-month retainer contracts. My monthly income averages
around $8,500, though it varies -- last December was lower ($4,200) because one client
was on holiday shutdown. I also do occasional Uber driving on weekends which brings in
roughly $300-400 per month extra.

I pay $2,100 per month in rent (always on time -- I can provide 24 months of receipts).
My main expenses are rent, utilities (~$180/month), streaming services, groceries, and
my gym membership. I don't have any credit card debt currently -- I closed my Westpac
credit card 8 months ago after paying it off in full.

I have $12,000 in savings (NAB account) which I've been building for two years. I'm
a bit worried about the next six months because one of my retainer contracts is up for
renewal in March, but I'm confident it will be renewed based on the client feedback I've
received. I've never missed a loan payment or bill payment in my life.
"""

# Stage 1: Income stability extraction
INCOME_PROMPT = """You are a credit analyst extracting income stability signals from an applicant narrative.
Extract the following as JSON:
- primary_income_type: EMPLOYED / SELF_EMPLOYED / GIG / MIXED
- monthly_income_estimate: estimated monthly income in AUD
- income_stability_score: 0.0 (very unstable) to 1.0 (very stable)
- income_risk_flags: list of risk factors mentioned (max 3 strings)
Return ONLY valid JSON, no markdown."""

# Stage 2: Spending behaviour
SPENDING_PROMPT = """You are a credit analyst extracting spending behaviour signals from an applicant narrative.
Extract the following as JSON:
- has_rental_history: true/false
- rental_payment_reliability: 0.0 to 1.0 (1.0 = always on time, documented)
- debt_management: PROACTIVE / ADEQUATE / CONCERNING
- savings_behaviour: 0.0 (no savings) to 1.0 (strong regular savings)
- discretionary_spend_flag: true if applicant mentions excessive discretionary spending
Return ONLY valid JSON, no markdown."""

# Stage 3: Financial stress indicators
STRESS_PROMPT = """You are a credit analyst identifying financial stress signals in an applicant narrative.
Extract the following as JSON:
- near_term_income_risk: true/false (any upcoming income uncertainty mentioned?)
- risk_description: one sentence describing the main risk, or null
- mitigating_factors: list of factors that reduce the risk (max 3 strings)
- overall_stress_level: LOW / MEDIUM / HIGH
Return ONLY valid JSON, no markdown."""

# Stage 4: Credit signal summary
SUMMARY_PROMPT = """You are a senior credit analyst. Given the following alternative data signals,
produce a structured credit recommendation.
Return JSON with:
- alternative_data_score: 0.0 (very risky) to 1.0 (very creditworthy)
- recommended_action: APPROVE / DECLINE / REFER_TO_ANALYST
- key_positive_signals: list of 3 positive factors
- key_risk_signals: list of 3 risk factors
- suggested_bureau_pull: true/false (whether traditional bureau check adds value)
Return ONLY valid JSON, no markdown."""

def call_gemini(model, prompt, context):
    """Call Gemini and parse JSON response."""
    response = model.generate_content(f"{prompt}\n\nApplicant narrative:\n{context}")
    text = response.text.strip()
    if text.startswith("```"):
        text = text.split("```")[1]
        if text.startswith("json"):
            text = text[4:]
    return json.loads(text.strip())

# Mock outputs (used when no API key is present)
MOCK_INCOME = {
    "primary_income_type": "MIXED",
    "monthly_income_estimate": 8800,
    "income_stability_score": 0.62,
    "income_risk_flags": [
        "Freelance retainer renewal risk (March)",
        "Income variability observed (Dec $4,200 vs avg $8,500)",
        "Gig income component (~$350/month Uber) - unguaranteed"
    ]
}
MOCK_SPENDING = {
    "has_rental_history": True,
    "rental_payment_reliability": 0.95,
    "debt_management": "PROACTIVE",
    "savings_behaviour": 0.72,
    "discretionary_spend_flag": False
}
MOCK_STRESS = {
    "near_term_income_risk": True,
    "risk_description": "One of two retainer contracts up for renewal in March; outcome uncertain.",
    "mitigating_factors": [
        "Strong client relationship mentioned",
        "$12,000 savings buffer provides 1.4 months coverage",
        "Second retainer provides partial income continuity"
    ],
    "overall_stress_level": "MEDIUM"
}
MOCK_SUMMARY = {
    "alternative_data_score": 0.71,
    "recommended_action": "REFER_TO_ANALYST",
    "key_positive_signals": [
        "24 months documented on-time rental payments (strong payment discipline)",
        "Proactive debt management: closed credit card after full payoff",
        "Consistent savings accumulation ($12K over 2 years)"
    ],
    "key_risk_signals": [
        "Self-employed with retainer renewal risk in near term (March)",
        "Income volatility: December dip to $4,200 vs $8,500 avg",
        "Gig income component creates instability floor"
    ],
    "suggested_bureau_pull": True
}

# Execute pipeline
if USE_LIVE_API:
    genai.configure(api_key=GEMINI_API_KEY)
    model_g = genai.GenerativeModel("gemini-1.5-flash",
                                     generation_config={"temperature": 0,
                                                        "response_mime_type": "application/json"})
    try:
        print("\n--- Live Gemini Pipeline ---")
        income_out  = call_gemini(model_g, INCOME_PROMPT,   APPLICANT_NARRATIVE)
        spending_out = call_gemini(model_g, SPENDING_PROMPT, APPLICANT_NARRATIVE)
        stress_out  = call_gemini(model_g, STRESS_PROMPT,   APPLICANT_NARRATIVE)
        ctx = json.dumps({"income": income_out, "spending": spending_out, "stress": stress_out})
        summary_out = call_gemini(model_g, SUMMARY_PROMPT, ctx)
    except Exception as e:
        print(f"Live API error ({e}) -- falling back to mock outputs.")
        income_out, spending_out, stress_out, summary_out = (
            MOCK_INCOME, MOCK_SPENDING, MOCK_STRESS, MOCK_SUMMARY)
else:
    income_out, spending_out, stress_out, summary_out = (
        MOCK_INCOME, MOCK_SPENDING, MOCK_STRESS, MOCK_SUMMARY)

# Print structured output
sep = "-" * 65
action_map = {"APPROVE": "APPROVE", "DECLINE": "DECLINE", "REFER_TO_ANALYST": "REFER"}
signal = action_map.get(summary_out.get("recommended_action", "?"), "?")
score  = summary_out.get("alternative_data_score", 0)
print(sep)
print("  ALTERNATIVE DATA CREDIT SIGNAL -- GEMINI 4-STAGE PIPELINE")
print(sep)
print(f"  Decision:               {signal}  (alt-data score: {score:.2f}/1.00)")
print(f"  DDF Station 1 (ETL):    Narrative -> structured alternative data")
print(sep)
print(f"  Stage 1 (Income):       {income_out.get('primary_income_type','')} | "
      f"AUD {income_out.get('monthly_income_estimate',0):,}/month | "
      f"Stability = {income_out.get('income_stability_score',0):.2f}")
for flag in income_out.get("income_risk_flags", []):
    print(f"    - {flag}")
print(f"\n  Stage 2 (Spending):     Rental reliability = {spending_out.get('rental_payment_reliability',0):.2f} | "
      f"Debt: {spending_out.get('debt_management','')} | "
      f"Savings: {spending_out.get('savings_behaviour',0):.2f}")
print(f"\n  Stage 3 (Stress):       {stress_out.get('overall_stress_level','')} stress | "
      f"Near-term risk: {'YES' if stress_out.get('near_term_income_risk') else 'NO'}")
if stress_out.get("risk_description"):
    print(f"    Risk: {stress_out['risk_description']}")
print(f"\n  Stage 4 (Recommendation):")
print("    Positives:")
for p in summary_out.get("key_positive_signals", []):
    print(f"      + {p}")
print("    Risks:")
for r in summary_out.get("key_risk_signals", []):
    print(f"      - {r}")
print(sep)
print("  SR 11-7 audit: Model=gemini-1.5-flash, T=0, alt-data pipeline v5.1")
print("  Human review required before credit decision. NCA s124 applies.")
print(sep)


---
## Week 5 Summary

| § | Topic | Key result |
|---|---|---|
| §1 | Logistic Scorecard (WoE/IV) | Feature IV ranking: credit_score and debt_to_income dominate; AUC ~0.72 |
| §2 | XGBoost + Alternative Data | Alt data adds ~6 Gini points; balance_stability and util_payment_score highest importance |
| §3 | MLP Neural Network | Non-linear model; minimal marginal gain over XGBoost; high SR 11-7 validation load |
| §4 | Model Comparison | XGBoost+alt: AUC~0.82, Gini~0.64, KS~48% — regulatory-optimal for Australian ADI |
| §5 | SHAP | Adverse action notice generation; SHAP satisfies SR 11-7 conceptual soundness |
| §6 | PSI Drift | Post-rate-rise: PSI>0.20 for income_k and debt_to_income — APRA CPS 220 retrain triggered |
| §7 | Gemini Pipeline | 4-stage extraction: income stability, spending behaviour, stress indicators, alt-data score |

**Key Week 5 insights:**
- Alternative data (open banking CDR) adds 20+ Gini points — the largest single performance lever in retail credit
- WoE/IV screening is a prerequisite for logistic scorecards and provides an interpretable feature selection framework for any model
- PSI is the primary production monitoring metric; APRA CPS 220 expects documented PSI thresholds and escalation procedures
- SHAP bridges the interpretability gap for non-linear models, enabling SR 11-7 compliance for XGBoost and neural networks
- LLM-based narrative extraction (Gemini pipeline) extends the information set beyond structured data, capturing qualitative risk signals invisible to open banking feeds

**Week 6 preview:** The Tech Lab applies the complete DDF pipeline to a live financial dataset — from raw data ingestion through feature engineering, model training, and monitoring dashboard construction.

---

## References

Arik, S.O., & Pfister, T. (2019). TabNet: Attentive interpretable tabular learning. *arXiv:1908.07442*.

Board of Governors of the Federal Reserve System (2011). *SR 11-7: Guidance on model risk management*. Federal Reserve.

Cheng, H.T., Koc, L., Harmsen, J., Shaked, T., Chandra, T., Aradhye, H., … Shah, H. (2016). Wide & deep learning for recommender systems. *Proceedings of the 1st Workshop on Deep Learning for Recommender Systems*, 7–10. ACM.

Hric, J., & Lin, Y. (2026). *FINS5557 applied AI in finance course notes: Week 5 — banking transformation*. UNSW Business School.

Klarna (2023). *Klarna AI assistant handles two-thirds of customer service chats in its first month*. OpenAI Case Studies.

Lundberg, S., & Lee, S.I. (2017). A unified approach to interpreting model predictions. *Advances in Neural Information Processing Systems*, 30, 4765–4774.

McKinsey & Company (2025). *The end of inertia: Agentic AI's disruption of retail and SME banking*. McKinsey Global Institute.

Plaid (2024). *Generative AI and synthetic identity fraud*. Plaid Resources.

Siddiqi, N. (2006). *Credit risk scorecards: Developing and implementing intelligent credit scoring*. Wiley.

Stripe (2025, May 7). *Stripe unveils AI foundation model for payments*. TechCrunch.

APRA (2023). *CPS 230: Operational risk management*. Australian Prudential Regulation Authority.

ASIC (2022). *Regulatory Guide 255: Providing digital financial product advice*. Australian Securities and Investments Commission.
